In [ ]:

import torch

print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("Nombre de GPU :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i} :", torch.cuda.get_device_name(i))
    print(f"Mémoire : {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} Go")



In [6]:
import sys, shutil, importlib.metadata as md
import torch
print("Python", sys.version.split()[0], "CUDA", torch.cuda.is_available())
print("GPU", [(torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/1024**3,1)) for i in range(torch.cuda.device_count())])
for package in ["transformers","heretic-llm","bitsandbytes","accelerate"]:
    try: print(package, md.version(package))
    except md.PackageNotFoundError: print(package, "absent")
for path in ["/kaggle/working", "/tmp", "/root/.cache"]:
    d=shutil.disk_usage(path)
    print(path, "libre Go", round(d.free/1024**3,2))

Python 3.13.15 CUDA True
GPU [('Tesla T4', 14.6), ('Tesla T4', 14.6)]
transformers 5.16.1
heretic-llm absent
bitsandbytes absent
accelerate 1.14.0
/kaggle/working libre Go 19.5
/tmp libre Go 1045.86
/root/.cache libre Go 1045.86


In [7]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","heretic-llm==1.4.0","bitsandbytes"],check=True)
print("Dépendances installées")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.4 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
bigframes 2.48.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.4 which is incompatible.


Dépendances installées


In [9]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","numpy>=2.1,<2.3","pillow<13"],check=True)
subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchao"],check=True)
subprocess.run([sys.executable,"-c","import heretic, bitsandbytes, transformers; print('IMPORTS OK'); import inspect, heretic.model; print(inspect.getsource(heretic.model.Model.__init__)[:11000])"],check=True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
bigframes 2.48.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
IMPORTS OK
    def __init__(self, settings: Settings):
        self.settings = settings
        self.needs_reload = False

        self.revision_kwargs = {}
        if settings.model_commit is not None:
            self.revision_kwargs["revision"] = settings.model_commit

        print()
        print(f"Loading model [bold]{settings.model}[/]...")

        self.tokenizer = AutoTokenizer.from_pretrained(
            settings.model,
            **self.revision_kwargs,
        )

        # Multimodal models have a processor we'll want to save.
        self.processor = None
        if get_model_class(settings.model) == AutoModelForImageTextToText:
            self.processor = AutoProcessor.from_pretrained(
                settings.model,
                **self.revision_kwargs,
            )

        # Fallback for tokenizers that don't declare a special pad token.
        if s

CompletedProcess(args=['/usr/bin/python3', '-c', "import heretic, bitsandbytes, transformers; print('IMPORTS OK'); import inspect, heretic.model; print(inspect.getsource(heretic.model.Model.__init__)[:11000])"], returncode=0)

In [10]:
from pathlib import Path
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient
upgrade_env = os.environ.copy()
upgrade_env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
upgrade_env["PYTHONUNBUFFERED"] = "1"
baseline_code = r'''
import json, torch
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig
MODEL_ID="Qwen/Qwen3.5-9B"
print("Chargement Qwen3.5-9B en 4 bits", flush=True)
tok=AutoTokenizer.from_pretrained(MODEL_ID)
net=AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto", attn_implementation="eager", quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.float16,bnb_4bit_quant_type="nf4"))
net.eval()
print("CHARGEMENT OK", flush=True)
questions=["Combien font 17 fois 23 ? Réponds en une phrase.","Propose trois slogans professionnels et positifs en français pour Kris Design, une agence de graphisme. Sans commentaire."]
resultats=[]
for q in questions:
    text=tok.apply_chat_template([{"role":"user","content":q}],tokenize=False,add_generation_prompt=True,enable_thinking=False)
    x=tok(text,return_tensors="pt").to(next(net.parameters()).device)
    with torch.inference_mode():
        y=net.generate(**x,max_new_tokens=180,do_sample=False,pad_token_id=tok.eos_token_id)
    a=tok.decode(y[0,x["input_ids"].shape[1]:],skip_special_tokens=True)
    print("QUESTION:",q,"\nREPONSE:",a,flush=True)
    resultats.append({"question":q,"reponse":a})
from pathlib import Path
Path("/kaggle/working/qwen35_baseline.json").write_text(json.dumps(resultats,ensure_ascii=False,indent=2))
print("BASELINE TERMINEE",flush=True)
'''
Path("/tmp/kris_qwen35_baseline.py").write_text(baseline_code)
subprocess.run([sys.executable,"-u","/tmp/kris_qwen35_baseline.py"],env=upgrade_env,check=True)

Chargement Qwen3.5-9B en 4 bits


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 760/760 [01:02<00:00, 12.18it/s]


CHARGEMENT OK
QUESTION: Combien font 17 fois 23 ? Réponds en une phrase. 
REPONSE: Le produit de 17 par 23 est égal à 391.

QUESTION: Propose trois slogans professionnels et positifs en français pour Kris Design, une agence de graphisme. Sans commentaire. 
REPONSE: Kris Design : Donnez vie à votre vision.
Kris Design : L'élégance au service de votre marque.
Kris Design : Créons ensemble votre identité.

BASELINE TERMINEE


CompletedProcess(args=['/usr/bin/python3', '-u', '/tmp/kris_qwen35_baseline.py'], returncode=0)

In [11]:
import subprocess, sys
inspect_code = "import inspect, heretic.main; s=inspect.getsource(heretic.main); a=s.index('Which trial do you want to use?'); print(s[a-2000:a+15000])"
subprocess.run([sys.executable,"-c",inspect_code],check=True)

est_trials
            ]

            choices.append(
                Choice(
                    title="Run additional trials",
                    value="continue",
                )
            )

            choices.append(
                Choice(
                    title="Exit program",
                    value="",
                )
            )

            print()
            print("[bold green]Optimization finished![/]")
            print()
            print(
                (
                    "The following trials resulted in Pareto optimal combinations of refusals and KL divergence. "
                    "After selecting a trial, you will be able to save the model, upload it to Hugging Face, "
                    "chat with it to test how well it works, or run standard benchmarks on it. "
                    "You can return to this menu later to select a different trial. "
                    "[yellow]Note that KL divergence values above 0.5 usually indicate significant

CompletedProcess(args=['/usr/bin/python3', '-c', "import inspect, heretic.main; s=inspect.getsource(heretic.main); a=s.index('Which trial do you want to use?'); print(s[a-2000:a+15000])"], returncode=0)

In [ ]:
from pathlib import Path
import subprocess, sys, os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
upgrade_env = os.environ.copy()
upgrade_env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
upgrade_env["PYTHONUNBUFFERED"] = "1"
# Vérifier la destination avant de lancer le calcul.
print("Compte de sauvegarde :", HfApi(token=upgrade_env["HF_TOKEN"]).whoami()["name"])
heretic_job = r'''
import sys, json
from pathlib import Path
import heretic.main as hm
OUT=Path("/kaggle/working/qwen3.5-9b-heretic-adapter")
selected=False
saved=False
chosen={}
def select_action(message, choices, **kwargs):
    global selected, saved, chosen
    def val(c): return getattr(c,"value",c)
    if "Which trial" in message:
        if selected: return ""
        trials=[val(c) for c in choices if hasattr(val(c),"user_attrs")]
        healthy=[t for t in trials if t.user_attrs.get("kl_divergence",99)<=0.5]
        if not healthy: raise RuntimeError("Aucun essai sous le seuil KL 0.5 : export annulé")
        best=min(healthy,key=lambda t:(t.user_attrs.get("refusals",999),t.user_attrs.get("kl_divergence",99)))
        chosen=dict(best.user_attrs)
        selected=True
        print("ESSAI RETENU:", json.dumps(chosen,default=str),flush=True)
        return best
    if "What do you want to do" in message:
        if saved: return ""
        saved=True
        return "Save the model to a local folder"
    if "How would you like to proceed" in message:
        return val(choices[0])
    raise RuntimeError("Choix interactif inattendu: "+message)
hm.prompt_select=select_action
hm.prompt_path=lambda *a,**k: str(OUT)
sys.argv=["heretic","--model=Qwen/Qwen3.5-9B","--quantization=bnb_4bit",'--dtypes='+json.dumps(["float16"]),"--batch-size=2","--max-response-length=64","--n-trials=3","--n-startup-trials=1","--seed=42","--study-checkpoint-dir=/kaggle/working/qwen35-study","--export-strategy=adapter"]
hm.main()
if not (OUT/"adapter_config.json").exists(): raise RuntimeError("Export non confirmé")
(OUT/"selection_metrics.json").write_text(json.dumps(chosen,default=str,ensure_ascii=False,indent=2))
from huggingface_hub import HfApi
api=HfApi()
repo=api.whoami()["name"]+"/qwen3.5-9b-heretic-adapter"
api.create_repo(repo,repo_type="model",private=True,exist_ok=True)
api.upload_folder(repo_id=repo,folder_path=str(OUT),repo_type="model")
print("SAUVEGARDE HF CONFIRMEE:",repo,api.list_repo_files(repo),flush=True)
'''
# La liste de types est passée en JSON à Heretic.
Path("/kaggle/working/kris_qwen35_heretic_job.py").write_text(heretic_job)
print("Lancement de 3 essais Heretic. Les points de reprise restent dans qwen35-study.",flush=True)
subprocess.run([sys.executable,"-u","/kaggle/working/kris_qwen35_heretic_job.py"],env=upgrade_env,check=True)

Compte de sauvegarde : Krisdesign1
Lancement de 3 essais Heretic. Les points de reprise restent dans qwen35-study.
█░█░█▀▀░█▀▄░█▀▀░▀█▀░█░█▀▀  v1.4.0
█▀█░█▀▀░█▀▄░█▀▀░░█░░█░█░░  https://heretic-project.org
▀░▀░▀▀▀░▀░▀░▀▀▀░░▀░░▀░▀▀▀  https://github.com/p-e-w/heretic

Detected 2 CUDA device(s) (29.12 GB total VRAM)
CUDA Version: 12.8
Driver Version: 580.178.04
* CUDA 0: Tesla T4 (14.56 GB)
* CUDA 1: Tesla T4 (14.56 GB)

Loading model Qwen/Qwen3.5-9B...
* Trying dtype float16...


Loading weights: 100%|██████████| 760/760 [00:56<00:00, 13.37it/s] 


* Quantized to 4-bit precision
* LoRA adapters initialized (target types: down_proj, o_proj, out_proj)
* Transformer model with 32 layers
* Abliterable components:
  * attn.o_proj: 32 modules total
  * mlp.down_proj: 32 modules total

Resident system RAM: 2.15 GB
Allocated GPU VRAM: 7.37 GB
Reserved GPU VRAM: 7.48 GB

Loading good prompts from mlabonne/harmless_alpaca...


Generating test split: 100%|██████████| 6265/6265 [00:00<00:00, 1320534.43 examples/s]


* 400 prompts loaded

Loading bad prompts from mlabonne/harmful_behaviors...


Generating test split: 100%|██████████| 104/104 [00:00<00:00, 57562.37 examples/s]


* 400 prompts loaded

Checking for common response prefix...


In [ ]:
# KRIS AI — charger le modèle Heretic après la fin des essais.
from pathlib import Path
import gc, json, html, torch
import ipywidgets as W
from IPython.display import display, HTML
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
BASE="Qwen/Qwen3.5-9B"
ADAPTER=Path("/kaggle/working/qwen3.5-9b-heretic-adapter")
if not (ADAPTER/"adapter_config.json").exists():
    raise RuntimeError("Heretic n’a pas encore terminé son export. Ne pas charger le modèle pendant les essais.")
print("Chargement de Kris AI — Qwen3.5-9B + Heretic",flush=True)
gc.collect()
torch.cuda.empty_cache()
kris_tokenizer=AutoTokenizer.from_pretrained(BASE)
kris_base=AutoModelForImageTextToText.from_pretrained(BASE,dtype=torch.float16,device_map="auto",attn_implementation="eager",quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.float16,bnb_4bit_quant_type="nf4"))
kris_model=PeftModel.from_pretrained(kris_base,str(ADAPTER))
kris_model.eval()
def kris_answer(messages, limit=250):
    text=kris_tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    inputs=kris_tokenizer(text,return_tensors="pt",truncation=True,max_length=4096).to(next(kris_model.parameters()).device)
    with torch.inference_mode():
        result=kris_model.generate(**inputs,max_new_tokens=limit,do_sample=False,pad_token_id=kris_tokenizer.eos_token_id)
    return kris_tokenizer.decode(result[0,inputs["input_ids"].shape[1]:],skip_special_tokens=True).strip()
checks=[]
for q in ["Combien font 17 fois 23 ? Réponds en une phrase.","Propose trois slogans professionnels et positifs en français pour Kris Design, une agence de graphisme. Sans commentaire."]:
    answer=kris_answer([{"role":"user","content":q}])
    checks.append({"question":q,"reponse":answer})
    print(q,"\n",answer,flush=True)
Path("/kaggle/working/qwen35_after_heretic.json").write_text(json.dumps(checks,ensure_ascii=False,indent=2))
assert "391" in checks[0]["reponse"], "Le calcul de contrôle a échoué : ne pas valider ce modèle."
api_kris=HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
repo_kris=api_kris.whoami()["name"]+"/qwen3.5-9b-heretic-adapter"
api_kris.upload_file(path_or_fileobj="/kaggle/working/qwen35_after_heretic.json",path_in_repo="tests_apres.json",repo_id=repo_kris)
api_kris.upload_file(path_or_fileobj="/kaggle/working/qwen35_baseline.json",path_in_repo="tests_avant.json",repo_id=repo_kris)
print("TESTS ET SAUVEGARDE TERMINÉS",flush=True)
messages_kris=[]
head=W.HTML('<div style="font:11px system-ui;letter-spacing:3px;color:#91a89c">ESPACE PERSONNEL</div><h2 style="font:600 30px system-ui;color:#eef2f5;margin:10px 0">Kris AI.</h2><p style="font:13px system-ui;color:#a5b0be">Qwen3.5-9B · Heretic · Hébergé sur Kaggle</p>')
chat=W.HTML('<p style="color:#a5b0be;font:14px system-ui">Bonjour Christenvie. Que souhaites-tu explorer ?</p>')
question=W.Textarea(placeholder="Écris ta question ici…",layout=W.Layout(width="100%",height="105px"))
send=W.Button(description="Envoyer",icon="arrow-right",button_style="success")
clear=W.Button(description="Nouvelle discussion")
length=W.Dropdown(options=[("Courte",160),("Standard",350),("Détaillée",700)],value=350,layout=W.Layout(width="170px"))
status=W.HTML('<small style="color:#94b49c">● Prêt</small>')
def draw_kris():
    chat.value='<div style="max-height:450px;overflow:auto">'+''.join('<div style="padding:16px 0;border-top:1px solid #303743"><div style="font:600 10px system-ui;letter-spacing:2px;color:#91a89c">'+('TOI' if m['role']=='user' else 'KRIS AI')+'</div><div style="white-space:pre-wrap;font:14px/1.7 system-ui;color:#e0e6ee;margin-top:8px">'+html.escape(m['content'])+'</div></div>' for m in messages_kris)+'</div>'
def ask_kris(_):
    q=question.value.strip()
    if not q:return
    send.disabled=clear.disabled=question.disabled=True
    messages_kris.append({"role":"user","content":q})
    draw_kris()
    status.value='<small style="color:#b6cbbd">Réflexion en cours…</small>'
    try:
        reply=kris_answer(messages_kris[-12:],length.value)
        messages_kris.append({"role":"assistant","content":reply})
        question.value=""
        draw_kris()
        status.value='<small style="color:#94b49c">● Prêt</small>'
    except Exception:
        messages_kris.pop()
        draw_kris()
        status.value='<small style="color:#dcaca5">La réponse a échoué. Réessaie avec une question plus courte.</small>'
    finally:send.disabled=clear.disabled=question.disabled=False
def reset_kris(_):
    messages_kris.clear()
    question.value=""
    draw_kris()
send.on_click(ask_kris)
clear.on_click(reset_kris)
panel=W.VBox([head,chat,question,W.HBox([send,clear,length],layout=W.Layout(gap="10px",flex_flow="row wrap")),status],layout=W.Layout(width="95%",max_width="850px",padding="28px",margin="20px auto",border="1px solid #303743"))
panel.add_class("kris-heretic-panel")
display(HTML('<style>.kris-heretic-panel{background:#151a21;border-radius:16px}.kris-heretic-panel textarea{background:#202630!important;color:#eef2f5!important;border:1px solid #39434f!important;border-radius:9px!important;padding:12px!important}.kris-heretic-panel button{border-radius:7px!important}</style>'))
display(panel)

In [ ]:
from pathlib import Path
p = Path("/kaggle/working")
for d in (p / "qwen-heretic", p / "checkpoints"):
    print(str(d), "existe :", d.exists())
    if d.exists():
        for f in d.iterdir():
            print(f.name, f.stat().st_size)

In [ ]:

%pip install -U heretic-llm



In [ ]:

import torch
from importlib.metadata import version

print("Heretic :", version("heretic-llm"))
print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("Nombre de GPU :", torch.cuda.device_count())


In [ ]:

!heretic --help


In [ ]:


!heretic --model=Qwen/Qwen2.5-1.5B-Instruct --n-trials=3 --n-startup-trials=1 --batch-size=2


In [ ]:

%pip uninstall -y torchao


In [ ]:

!python -c "import importlib.util; print('torchao présent :', importlib.util.find_spec('torchao') is not None)"



In [ ]:
# !heretic --model=Qwen/Qwen2.5-1.5B-Instruct --n-trials=3 --n-startup-trials=1 --batch-size=2


In [ ]:
!ls -lh /kaggle/working/checkpoints/


In [ ]:

!cd /kaggle/working && printf '1\n1\n1\n/kaggle/working/qwen-heretic\n1\n5\n3\n' | heretic --model=Qwen/Qwen2.5-1.5B-Instruct --n-trials=3 --n-startup-trials=1 --batch-size=2


In [ ]:

import os

dossier = "/kaggle/working/qwen-heretic"

print("Dossier présent :", os.path.isdir(dossier))

if os.path.isdir(dossier):
    for fichier in os.listdir(dossier):
        chemin = os.path.join(dossier, fichier)
        if os.path.isfile(chemin):
            taille = os.path.getsize(chemin) / (1024**2)
            print(f"{fichier} : {taille:.2f} Mo")


In [ ]:

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)

username = api.whoami()["name"]
repo_id = f"{username}/qwen2.5-1.5b-heretic-trial2"

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True,
    exist_ok=True
)

api.upload_folder(
    folder_path="/kaggle/working/qwen-heretic",
    repo_id=repo_id,
    repo_type="model"
)

print("Modèle sauvegardé :", f"https://huggingface.co/{repo_id}")
print("Fichiers vérifiés :", api.list_repo_files(repo_id))


In [ ]:
import torch
import transformers
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
print("PyTorch :", torch.__version__)
print("Transformers :", transformers.__version__)
print("GPU disponible :", torch.cuda.is_available())
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
model_id = "Krisdesign1/qwen2.5-1.5b-heretic-trial2"
model_info = HfApi(token=hf_token).model_info(model_id)
print("Dépôt accessible :", model_info.id)
print("Fichiers :", [f.rfilename for f in model_info.siblings])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
print("Chargement du modèle sauvegardé...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id, token=hf_token, dtype=torch.float16, device_map="auto"
)
model.eval()
print("MODÈLE PRÊT :", model_id, flush=True)

def repondre(question, max_tokens=180):
    messages = [
        {"role": "system", "content": "Tu es un assistant utile. Réponds en français avec clarté et précision."},
        {"role": "user", "content": question}
    ]
    texte = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(texte, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

questions = [
    "Explique en trois phrases simples ce qu’est une intelligence artificielle.",
    "Combien font 17 × 23 ? Donne le résultat et une courte vérification.",
    "Propose trois slogans courts pour une entreprise de graphisme appelée Kris Design."
]
for question in questions:
    print("\nQUESTION :", question, flush=True)
    print("RÉPONSE :", repondre(question), flush=True)
print("\nTESTS TERMINÉS", flush=True)

In [ ]:
# Modifie la question entre guillemets, puis exécute uniquement cette cellule.
ma_question = "Bonjour ! Présente-toi en deux phrases."
print(repondre(ma_question, max_tokens=300))

In [ ]:
# KRIS AI — interface. Exécuter cette cellule pour ouvrir le chat.
import html
import torch
import ipywidgets as widgets
from IPython.display import display, HTML
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient

MODEL_ID = "Krisdesign1/qwen2.5-1.5b-heretic-trial2"
if "model" not in globals() or "tokenizer" not in globals():
    print("Chargement du modèle…")
    _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=_hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, token=_hf_token, dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto")
    model.eval()
    del _hf_token

display(HTML("""
<style>
.kris-panel {max-width:860px!important;margin:20px auto!important;padding:28px!important;
background:#11151b!important;border:1px solid #2a313b!important;border-radius:18px!important;color:#edf0f5!important}
.kris-panel textarea {background:#191f28!important;color:#edf0f5!important;border:1px solid #384250!important;
border-radius:10px!important;padding:14px!important;font:14px/1.6 system-ui!important;resize:vertical!important}
.kris-panel .widget-button {border-radius:8px!important;font:600 13px system-ui!important;box-shadow:none!important}
.kris-send {background:#d8e9dd!important;color:#172b20!important;border:0!important}
.kris-clear {background:#222a35!important;color:#d3dbe5!important;border:1px solid #384250!important}
.kris-panel .widget-label {color:#aebac8!important}
</style>
"""))
historique_kris = []
entete_kris = widgets.HTML("""
<div style="font-family:system-ui">
<div style="font-size:11px;letter-spacing:3px;color:#9baaac;margin-bottom:9px">ESPACE PERSONNEL</div>
<div style="font-size:29px;font-weight:650;letter-spacing:-1px;color:#f2f4f8">Kris AI<span style="color:#9ebaab">.</span></div>
<p style="font-size:13px;color:#a7b3c2;margin:8px 0 22px">Une question. Un espace pour réfléchir.</p></div>""")
conversation_kris = widgets.HTML()
question_kris = widgets.Textarea(placeholder="Écris ta question ici…",
    layout=widgets.Layout(width="100%",height="110px"))
envoyer_kris = widgets.Button(description="Envoyer",icon="arrow-right",
    layout=widgets.Layout(width="130px",height="38px"))
envoyer_kris.add_class("kris-send")
effacer_kris = widgets.Button(description="Nouvelle discussion",
    layout=widgets.Layout(width="180px",height="38px"))
effacer_kris.add_class("kris-clear")
longueur_kris = widgets.Dropdown(options=[("Courte",160),("Standard",350),("Détaillée",700)],
    value=350,description="Réponse :",layout=widgets.Layout(width="220px"))
statut_kris = widgets.HTML()
def afficher_kris():
    if not historique_kris:
        conversation_kris.value = '<div style="font:14px/1.7 system-ui;color:#9dabbc;padding:24px 0 30px">Bonjour Christenvie.<br>Que souhaites-tu explorer aujourd’hui ?</div>'
    else:
        cartes = []
        for message in historique_kris:
            auteur = "TOI" if message["role"] == "user" else "KRIS AI"
            couleur = "#9ebaab" if message["role"] == "assistant" else "#96a5b9"
            cartes.append(f'<div style="padding:17px 0;border-top:1px solid #2a313b"><div style="font:600 10px system-ui;letter-spacing:1.5px;color:{couleur};margin-bottom:8px">{auteur}</div><div style="white-space:pre-wrap;overflow-wrap:anywhere;font:14px/1.75 system-ui;color:#e2e7ee">{html.escape(message["content"])}</div></div>')
        conversation_kris.value = '<div style="max-height:490px;overflow-y:auto;padding-right:12px;margin-bottom:18px">' + "".join(cartes) + "</div>"
    statut_kris.value = '<div style="font:11px system-ui;color:#829285;margin-top:12px">● Prêt · Qwen 1.5B · Session Kaggle</div>'
def envoyer_question_kris(_):
    question = question_kris.value.strip()
    if not question:
        return
    envoyer_kris.disabled = effacer_kris.disabled = question_kris.disabled = True
    historique_kris.append({"role":"user","content":question})
    afficher_kris()
    statut_kris.value = '<div style="font:12px system-ui;color:#c7d8cb;margin-top:12px">Réflexion en cours…</div>'
    try:
        messages = [{"role":"system","content":"Tu es Kris AI, un assistant utile. Réponds en français avec clarté et précision. Si tu ne sais pas, dis-le. Ne prétends pas avoir accès à Internet."}] + historique_kris[-12:]
        texte = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
        entrees = tokenizer(texte,return_tensors="pt",truncation=True,max_length=6000).to(model.device)
        with torch.inference_mode():
            sortie = model.generate(**entrees,max_new_tokens=longueur_kris.value,
                do_sample=False,pad_token_id=tokenizer.eos_token_id)
        reponse = tokenizer.decode(sortie[0,entrees["input_ids"].shape[1]:],skip_special_tokens=True).strip()
        historique_kris.append({"role":"assistant","content":reponse or "Aucune réponse générée. Reformule ta question."})
        question_kris.value = ""
        afficher_kris()
    except Exception:
        historique_kris.pop()
        afficher_kris()
        statut_kris.value = '<div style="font:12px system-ui;color:#e2b7a9;margin-top:12px">La réponse a échoué. Réessaie ou relance cette cellule si la session a été interrompue.</div>'
    finally:
        envoyer_kris.disabled = effacer_kris.disabled = question_kris.disabled = False
def nouvelle_discussion_kris(_):
    historique_kris.clear()
    question_kris.value = ""
    afficher_kris()
envoyer_kris.on_click(envoyer_question_kris)
effacer_kris.on_click(nouvelle_discussion_kris)
afficher_kris()
panneau_kris = widgets.VBox([entete_kris,conversation_kris,question_kris,
    widgets.HBox([envoyer_kris,effacer_kris,longueur_kris],layout=widgets.Layout(gap="10px",flex_flow="row wrap",margin="12px 0 0 0")),
    statut_kris],layout=widgets.Layout(width="100%"))
panneau_kris.add_class("kris-panel")
display(panneau_kris)
